### Feature Engineering and Regression

In [1]:
!pip install xgboost
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder,LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import xgboost as xgb

print('Libraries are Ready')

Libraries are Ready


#### Merging in indicator_scale

In [2]:
gap_df = pd.read_csv('inequality_gap_sex.csv')

if 'indicator_scale' in gap_df.columns and 'whoreg6' in gap_df.columns:
    merged = gap_df.copy()
else:
    try:
        df_full = pd.read_csv('who_data_cleaned.csv')
    except Exception:
        df_full = pd.read_excel('WHO Dataset.xlsx', sheet_name='Data')
    merged = gap_df.merge(df_full[['indicator_abbr', 'indicator_scale']].drop_duplicates('indicator_abbr'), on='indicator_abbr', how='left')
    region_map = df_full[['iso3', 'whoreg6']].drop_duplicates('iso3')
    merged = merged.merge(region_map, on='iso3', how='left')

print("Merged shape:", merged.shape)
print("Missing indicator_scale:", merged['indicator_scale'].isnull().sum())
print("Missing whoreg6:", merged['whoreg6'].isnull().sum())

Merged shape: (79448, 14)
Missing indicator_scale: 0
Missing whoreg6: 9


#### Confirming What "No Target-Derived Information" Actually Means Here

In [3]:
print("Target: inequality_gap = female_estimate - male_estimate")
print("\nColumns that MUST be excluded from features:")
print("  - female_estimate  (directly builds the target)")
print("  - male_estimate    (directly builds the target)")
print("\nColumns that are SAFE to use as features:")
print("  - date, whoreg6, wbincome2025, indicator_scale, favourable_indicator")

Target: inequality_gap = female_estimate - male_estimate

Columns that MUST be excluded from features:
  - female_estimate  (directly builds the target)
  - male_estimate    (directly builds the target)

Columns that are SAFE to use as features:
  - date, whoreg6, wbincome2025, indicator_scale, favourable_indicator


This code explicitly states, in code and comments, exactly which columns are off-limits and why.

This is the single most important rule in this module, as `female_estimate` and `male_estimate` are literally the two numbers subtracted to create `inequality_gap` including either one as a feature would let the model cheat by nearly reconstructing the answer directly, rather than learning any genuine pattern. 
To stop the risk of leakage.

#### Handling the Missing Region and Income Group

In [4]:
print("Missing whoreg6:", merged['whoreg6'].isnull().sum())
print("Missing wbincome2025:", merged['wbincome2025'].isnull().sum())

before = len(merged)
model_df = merged.dropna(subset=['whoreg6', 'wbincome2025']).copy()
print(f"\nRows before: {before}, after: {len(model_df)}")
print(f"Dropped: {before - len(model_df)} ({(before-len(model_df))/before*100:.2f}%)")

Missing whoreg6: 9
Missing wbincome2025: 792

Rows before: 79448, after: 78647
Dropped: 801 (1.01%)


These are the two features which are essential context for the model, and there's no defensible way to invent a WHO region or income classification for a row that doesn't have one .These are typically non-standard territories, echoing the Faroe Islands finding from Module 1. 
Dropping barely 1% of rows is a small, honest cost for keeping the remaining features fully populated.

#### Encoding the Income Group (Ordinal as it Has a Natural Order)

In [5]:
income_order = ['Low-income','Lower-middle-income','Upper-middle-income','High-income']
income_encoder = OrdinalEncoder(categories = [income_order])
model_df['income_encoded'] = income_encoder.fit_transform(model_df[['wbincome2025']])

print(model_df[['wbincome2025','income_encoded']].drop_duplicates().sort_values('income_encoded'))
print('\n Encoding completed')

             wbincome2025  income_encoded
0              Low-income             0.0
1120  Lower-middle-income             1.0
231   Upper-middle-income             2.0
952           High-income             3.0

 Encoding completed


Coverted income group into numbers 0–3, in genuine economic order.

##### The reason we chose OrdinalEncoder and not LabelEncoder here because income has a real, meaningful progression Low-income is genuinely "less than" Lower-middle-income. OrdinalEncoder with an explicit categories list guarantees this exact order. 
##### A LabelEncoder would assign numbers alphabetically instead, which would nonsensically place "High-income" before "Low-income."

#### Encoding the WHO Regions (No Natural Order)

In [6]:
region_encoder = LabelEncoder()
model_df['region_encoded'] = region_encoder.fit_transform(model_df['whoreg6'])
print(model_df[['whoreg6','region_encoded']].drop_duplicates().sort_values('region_encoded'))
print('\nConverted the 6 WHO regions into numbers 0–5.')

                    whoreg6  region_encoded
673                 African               0
1353               Americas               1
0     Eastern Mediterranean               2
231                European               3
4940        South-East Asia               4
2508        Western Pacific               5

Converted the 6 WHO regions into numbers 0–5.


##### The reason LabelEncoder is fine here, beacsue unlike for income there is no meaningful order between African and European they're just six distinct categories. 
##### Since our tree-based models (Random Forest, XGBoost) split on thresholds rather than treating the number as a true quantity, and Linear Regression's coefficient on this column is a secondary, less critical feature, this simpler encoding is an acceptable trade-off here rather than expanding to six separate one-hot columns.

#### Define Features and Target, Confirming No Leakage

In [7]:
feature_cols = ['date','region_encoded','income_encoded','indicator_scale','favourable_indicator']
X = model_df[feature_cols]
y = model_df['inequality_gap']

print("Feature columns:", feature_cols)
print("X shape:", X.shape)
print("\nLeakage check — are these in X?")
print("  female_estimate:", 'female_estimate' in X.columns)
print("  male_estimate:  ", 'male_estimate' in X.columns)

Feature columns: ['date', 'region_encoded', 'income_encoded', 'indicator_scale', 'favourable_indicator']
X shape: (78647, 5)

Leakage check — are these in X?
  female_estimate: False
  male_estimate:   False


#### Train/Test Split

In [8]:
cutoff_year = 2020
train_mask = X['date'] < cutoff_year
X_train, X_test = X[train_mask],X[~train_mask]
y_train, y_test = y[train_mask],y[~train_mask]

print(f'Train: {len(X_train)} rows ({len(X_train)/len(X)*100:.1f}%) — years 1998-2019')
print(f"Test:  {len(X_test)} rows ({len(X_test)/len(X)*100:.1f}%) — years 2020–2024")

Train: 59144 rows (75.2%) — years 1998-2019
Test:  19503 rows (24.8%) — years 2020–2024


##### the same country appears repeatedly across many years in this dataset. A random split could put, say, "France 2015" in training and "France 2016" in testing — two very similar rows — letting the model appear to perform well simply by having seen something nearly identical already. Splitting by year forces every model to predict genuinely unseen future years, which is a fair, honest test of whether it learned a real pattern.

#### Evaluation Function

In [9]:
def evaluate(y_true,y_pred,name):
    mae = mean_absolute_error(y_true,y_pred)
    rmse = np.sqrt(mean_squared_error(y_true,y_pred))
    r2 = r2_score(y_true,y_pred)
    print(f'{name:<25} MAE = {mae:>10.3f} RMSE = {rmse:>10.3f} R2 = {r2:>8.4f}')
    return {'model':name, 'MAE':mae, 'RMSE':rmse, 'R2':r2}

##### It wraps all three required metrics into one reusable function, so every model is scored identically and printed in a consistent format.
##### writing this once and calling it four times guarantees every model is judged by the exact same yardstick no risk of accidentally calculating one model's RMSE slightly differently from another's.

#### Training All Four Models on the Raw Target

In [11]:
results_v1 = []

baseline = DummyRegressor(strategy='mean')
baseline.fit(X_train, y_train)
results_v1.append(evaluate(y_test, baseline.predict(X_test), "Baseline (mean)"))

lr = LinearRegression()
lr.fit(X_train, y_train)
results_v1.append(evaluate(y_test, lr.predict(X_test), "Linear Regression"))

rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
results_v1.append(evaluate(y_test, rf.predict(X_test), "Random Forest"))

xgb_model = xgb.XGBRegressor(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)
results_v1.append(evaluate(y_test, xgb_model.predict(X_test), "XGBoost"))

Baseline (mean)           MAE =     24.158 RMSE =    111.927 R2 =  -0.0150
Linear Regression         MAE =     31.748 RMSE =    116.140 R2 =  -0.0928
Random Forest             MAE =     12.298 RMSE =    147.055 R2 =  -0.7520
XGBoost                   MAE =     12.708 RMSE =    164.311 R2 =  -1.1874


##### It trains and evaluates exactly the four models as synopsis specifies, on the target exactly as Module 3 defined it.
##### R2 is negative it needs investigating, not hiding an R² below 0 means the model's predictions are worse than simply always guessing the training set's average value. Random Forest and XGBoost are actually the worst performers by RMSE here, even though their MAE looks better a strong sign they're overfitting to extreme values in training and making occasional wildly-wrong predictions on test outliers, which RMSE punishes heavily (since it squares errors before averaging).

In [12]:
indicator_means = merged.groupby('indicator_abbr')['inequality_gap'].transform('mean')
ss_total = ((merged['inequality_gap'] - merged['inequality_gap'].mean())**2).sum()
ss_explained = ((indicator_means - merged['inequality_gap'].mean())**2).sum()

print(f"R2 if we predicted using each indicator's own average gap: {ss_explained/ss_total:.4f}")

R2 if we predicted using each indicator's own average gap: 0.0224


#### Applying Module 4's Own Trimming Methodology

In [13]:
p01 = model_df['inequality_gap'].quantile(0.01)
p99 = model_df['inequality_gap'].quantile(0.99)
model_df_trimmed = model_df[
    (model_df['inequality_gap'] >= p01) & (model_df['inequality_gap'] <= p99)
    ].copy()
print(f'Rows before trim: {len(model_df)}, after: {len(model_df_trimmed)}'
      f'({len(model_df_trimmed)/len(model_df)*100:.1f}%)')

Rows before trim: 78647, after: 77073(98.0%)


##### It removes the same top and bottom 1% of inequality_gap values that Module 4 already identified and excluded from its charts — the exact same methodology, now applied consistently to modeling.

#### Rebuilding Features and Split on the Trimmed Data

In [14]:
income_encoder2 = OrdinalEncoder(categories=[income_order])
model_df_trimmed['income_encoded'] = income_encoder2.fit_transform(model_df_trimmed[['wbincome2025']])
region_encoder2 = LabelEncoder()
model_df_trimmed['region_encoded'] = region_encoder2.fit_transform(model_df_trimmed['whoreg6'])

X2 = model_df_trimmed[feature_cols]
y2 = model_df_trimmed['inequality_gap']

train_mask2 = X2['date'] < cutoff_year
X_train2, X_test2 = X2[train_mask2], X2[~train_mask2]
y_train2, y_test2 = y2[train_mask2], y2[~train_mask2]

print(f"Train: {len(X_train2)} | Test: {len(X_test2)}")

Train: 57890 | Test: 19183


##### The encoders are refit here rather than reused, since the trimmed dataset is technically a different set of rows refitting on the correct data each time is good practice, even though the category lists themselves don't change.

#### Re-Run All Four Models

In [15]:
results_v2 = []

baseline2 = DummyRegressor(strategy='mean')
baseline2.fit(X_train2, y_train2)
results_v2.append(evaluate(y_test2, baseline2.predict(X_test2), "Baseline (mean)"))

lr2 = LinearRegression()
lr2.fit(X_train2, y_train2)
results_v2.append(evaluate(y_test2, lr2.predict(X_test2), "Linear Regression"))

rf2 = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf2.fit(X_train2, y_train2)
results_v2.append(evaluate(y_test2, rf2.predict(X_test2), "Random Forest"))

xgb_model2 = xgb.XGBRegressor(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)
xgb_model2.fit(X_train2, y_train2)
results_v2.append(evaluate(y_test2, xgb_model2.predict(X_test2), "XGBoost"))

Baseline (mean)           MAE =      4.553 RMSE =      8.254 R2 =  -0.0033
Linear Regression         MAE =      4.430 RMSE =      8.109 R2 =   0.0317
Random Forest             MAE =      4.410 RMSE =      7.988 R2 =   0.0603
XGBoost                   MAE =      4.429 RMSE =      7.997 R2 =   0.0582


##### As we can see the exact same four models, trained and tested the exact same way — only the outlier handling changed.


In [16]:
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf2.feature_importances_
}).sort_values('Importance', ascending=False)

print(importance_df)

                Feature  Importance
3       indicator_scale    0.403561
0                  date    0.201440
1        region_encoded    0.172107
2        income_encoded    0.144392
4  favourable_indicator    0.078501


##### shows which of the five features the best-performing model relied on most.
##### indicator_scale is the single most important feature at 40%, confirming from a completely different angle exactly what Module 4 discovered by investigating the extreme skew. The same variable that explained why the target was hard to model also turns out to be the most useful thing we have for modeling it. This is a coherent, connected finding across two modules, not two unrelated observations.